# VRC Compact59 — Colab GPU 学習

**「ランタイム」→「ランタイムのタイプを変更」で GPU を選択し、「すべてのセルを実行」してください。**

入力は `VRC_Compact59_Colab_7fe4fac1.zip`。最後に作られる **`VRC_Compact59_Colab_Result_7fe4fac1.zip`** を、元のChatGPTチャットへその名前のままアップロードしてください。

公開画像で59点の軽量候補を学習し、独立画像で精度を評価します。Hugging Faceへの公開と最終的なヘッドレスNode・VRM検証は、返却後にGitHub Actionsで行います。HF_TOKEN・GitHub認証・Driveマウントは不要です。


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, io, json, os, stat, subprocess, sys, zipfile

KIT_NAME = "VRC_Compact59_Colab_7fe4fac1.zip"
ROOT_NAME = "VRC_Compact59_Colab_7fe4fac1"
CONTENT = Path("/content")
ROOT = CONTENT / ROOT_NAME
os.chdir(CONTENT)
print("入力ZIPを選択してください:", KIT_NAME)
uploaded = files.upload()
if len(uploaded) != 1 or KIT_NAME not in uploaded:
    raise ValueError("指定名のZIPを1個だけアップロードしてください: " + KIT_NAME)
payload = uploaded[KIT_NAME]
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    members = archive.infolist()
    if len(members) > 100 or sum(m.file_size for m in members) > 128 * 1024 * 1024:
        raise ValueError("学習キットのファイル数またはサイズが上限を超えています")
    for member in members:
        path = PurePosixPath(member.filename)
        if path.is_absolute() or ".." in path.parts or not path.parts or path.parts[0] != ROOT_NAME:
            raise ValueError("学習キット外への展開は許可されていません")
        if stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("シンボリックリンクは受け付けません")
    bundle = json.loads(archive.read(ROOT_NAME + "/bundle.json"))
    if bundle.get("handoffId") != "7fe4fac1" or bundle.get("privateInputsIncluded") is not False:
        raise ValueError("このチャット向けの学習キットではありません")
    expected = {ROOT_NAME + "/" + entry["path"]: entry for entry in bundle["files"]}
    actual = {m.filename for m in members if not m.is_dir()} - {ROOT_NAME + "/bundle.json"}
    if actual != set(expected):
        raise ValueError("キットのファイル一覧が一致しません")
    for name, entry in expected.items():
        data = archive.read(name)
        if len(data) != entry["bytes"] or hashlib.sha256(data).hexdigest() != entry["sha256"]:
            raise ValueError("キットのSHA-256が一致しません: " + name)
    archive.extractall(CONTENT)
print("入力確認完了:", hashlib.sha256(payload).hexdigest())
del uploaded, payload


## 依存関係

ColabのCUDA版PyTorchを使います。追加の数値ライブラリは専用Python環境へ入れ、ノートブック本体のライブラリを上書きしません。


In [ ]:
ENV = ROOT / ".venv"
subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(ENV)], check=True)
PYTHON = str(ENV / "bin" / "python")
subprocess.run([PYTHON, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(ROOT / "colab" / "requirements.txt")], check=True)


## GPU学習と評価

2候補を4000 stepずつ学習し、元解像度を保つINT8候補も2種類比較します。学習768画像、候補選択128画像、最終test192画像。量子化の校正は学習集合の128画像だけを使います。過去の実験・受け渡し確認に使った画像は新しいvalidation/testから除きます。GPUが利用できなければ停止理由を結果へ記録します。

精度基準に達しない候補は、その事実を記録したまま返します。速度だけで採用モデルにはしません。


In [ ]:
TRAIN_STEPS = 4000
BATCH_SIZE = 32
subprocess.run([PYTHON, str(ROOT / "colab" / "run.py"), "--root", str(ROOT),
                "--steps", str(TRAIN_STEPS), "--batch-size", str(BATCH_SIZE)], check=True)


## 結果ZIPをダウンロード

このZIPを、元のChatGPTチャットへ **`VRC_Compact59_Colab_Result_7fe4fac1.zip`** のまま添付してください。モデルと集計結果だけを含め、画像・動画・特徴キャッシュは含めません。


In [ ]:
subprocess.run([PYTHON, str(ROOT / "colab" / "package_result.py"), "--root", str(ROOT), "--destination", str(CONTENT)], check=True)
RESULT = CONTENT / "VRC_Compact59_Colab_Result_7fe4fac1.zip"
run_status = json.loads((ROOT / "output" / "run-status.json").read_text())
print("実行結果:", run_status["status"])
print("返却するファイル:", RESULT.name)
files.download(str(RESULT))
